In [1]:
# -*- coding: utf-8 -*-
"""
SteganoGAN Training Script
Train a steganography model with tqdm progress bars and automatic metrics logging.
"""

import os
os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

import json
from time import time
import torch

from models import SteganoGAN
from decoders import DenseDecoder, BasicDecoder
from encoders import DenseEncoder, BasicEncoder, ResidualEncoder
from loader import DataLoader


In [ ]:
def main():
    """Main training function."""
    # Training configuration
    CONFIG = {
        'gpu': True,
        'data_depth': 1,
        'encoder': 'dense',  # Options: 'basic', 'residual', 'dense'
        'decoder': 'dense',  # Options: 'basic', 'dense'
        'epochs': 1,
        'batch_size': 4,  # Reduced for MPS memory constraints # 4
        'num_workers': 0, # Set to 0 for macOS/MPS compatibility # 8
        'dataset': 'div2k',
        'training_type': 'panet_dense'
    }

    # Set random seed for reproducibility
    torch.manual_seed(42)

    print("="*60)
    print("SteganoGAN Training")
    print("="*60)
    print("\nConfiguration:")
    for key, value in CONFIG.items():
        print(f"  {key}: {value}")
    print()

    # Select encoder and decoder based on config
    encoder_map = {
        'basic': BasicEncoder,
        'residual': ResidualEncoder,
        'dense': DenseEncoder
    }

    decoder_map = {
        'basic': BasicDecoder,
        'dense': DenseDecoder
    }

    encoder_class = encoder_map[CONFIG['encoder']]
    decoder_class = decoder_map[CONFIG['decoder']]

    # Create data loaders
    print("Loading datasets...")
    train = DataLoader(
        os.path.abspath('/Users/dmitryhoma/Projects/datasets/div2k/train'),
        batch_size=CONFIG['batch_size'],
        num_workers=CONFIG['num_workers'],
        shuffle=True
    )

    validation = DataLoader(
        os.path.abspath('/Users/dmitryhoma/Projects/datasets/div2k/val'),
        batch_size=CONFIG['batch_size'],
        num_workers=CONFIG['num_workers'],
        shuffle=False
    )

    print(f"Train dataset size: {len(train.dataset)}")
    print(f"Validation dataset size: {len(validation.dataset)}")

    # Create output directory
    timestamp = str(int(time()))
    log_dir = os.path.join('models', CONFIG['training_type'], timestamp)
    os.makedirs(log_dir, exist_ok=True)

    # Save configuration
    config_path = os.path.join(log_dir, "config.json")
    with open(config_path, "w") as f:
        json.dump(CONFIG, f, indent=2)

    print(f"\nLogs and checkpoints will be saved to: {log_dir}")

    # Initialize SteganoGAN model
    print(f"\nInitializing model with {CONFIG['encoder']} encoder and {CONFIG['decoder']} decoder...")
    steganogan = SteganoGAN(
        data_depth=CONFIG['data_depth'],
        encoder=encoder_class,
        decoder=decoder_class,
        gpu=CONFIG['gpu'],
        verbose=True,
        log_dir=log_dir
    )

    # Train the model
    print("\nStarting training...")
    steganogan.fit(train, validation, epochs=CONFIG['epochs'])

    # Save final model
    final_weights_path = os.path.join(log_dir, "weights.steg")
    steganogan.save(final_weights_path)

    print("\n" + "="*60)
    print("Training Complete!")
    print("="*60)
    print(f"Final model saved to: {final_weights_path}")
    print(f"Metrics saved to: {os.path.join(log_dir, 'metrics.log')}")

    # Display final metrics
    if steganogan.fit_metrics:
        print("\nFinal Epoch Metrics:")
        for key, value in steganogan.fit_metrics.items():
            if isinstance(value, float):
                print(f"  {key}: {value:.6f}")
            else:
                print(f"  {key}: {value}")


if __name__ == '__main__':
    main()

SteganoGAN Training

Configuration:
  gpu: True
  data_depth: 1
  encoder: dense
  decoder: dense
  epochs: 1
  batch_size: 8
  num_workers: 4
  dataset: div2k
  training_type: panet_dense

Loading datasets...
Train dataset size: 100
Validation dataset size: 100

Logs and checkpoints will be saved to: models/panet_dense/1770245917

Initializing model with dense encoder and dense decoder...
Using Apple GPU (MPS).

Starting training...

Epoch 1/1


Validation: 100%|██████████| 13/13 [00:32<00:00]



Epoch 1 Metrics:
  Train - Enc MSE: 0.174400, Dec Loss: 0.6975, Dec Acc: 0.5401
  Val   - Enc MSE: 0.096296, Dec Loss: 0.6877, Dec Acc: 0.5663
  Val   - SSIM: 0.2828, PSNR: 16.23, RSBPP: 0.1327
Model saved to models/panet_dense/1770245917/1.rsbpp-0.132673.p
Model saved to models/panet_dense/1770245917/weights.steg

Training Complete!
Final model saved to: models/panet_dense/1770245917/weights.steg
Metrics saved to: models/panet_dense/1770245917/metrics.log

Final Epoch Metrics:
  val.encoder_mse: 0.096296
  val.decoder_loss: 0.687681
  val.decoder_acc: 0.566336
  val.ssim: 0.282823
  val.psnr: 16.229786
  val.rsbpp: 0.132673
  train.encoder_mse: 0.174400
  train.decoder_loss: 0.697470
  train.decoder_acc: 0.540067
  epoch: 1
